In [349]:
# Import necessary libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import numpy as np

print("Libraries imported successfully.")


Libraries imported successfully.


In [350]:
import requests
from shapely.geometry import Point

# Overpass query for Spätis & corner stores in Berlin
query = """
[out:json];
(
  node[shop=convenience](52.4,13.2,52.7,13.6);
  node[shop=kiosk](52.4,13.2,52.7,13.6);
);
out body;
"""

# Send request
url = "https://overpass-api.de/api/interpreter"
resp = requests.post(url, data=query, timeout=180)
resp.raise_for_status()
data = resp.json()

# Normalize JSON to pandas dataframe
elements = data.get("elements", [])
df = pd.json_normalize(elements)

# Ensure lat/lon exist
df = df[df['lat'].notna() & df['lon'].notna()]

# Convert to GeoDataFrame
spatis_gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df['lon'].astype(float), df['lat'].astype(float))],
    crs="EPSG:4326"
)

# Minimal cleanup and renaming to match schema
spatis = spatis_gdf.rename(columns={
    'tags.name': 'name',
    'tags.brand': 'brand',
    'tags.operator': 'operator',
    'tags.opening_hours': 'opening_hours',
    'tags.phone': 'phone',
    'tags.website': 'website',
    'tags.source': 'source',
    'lat': 'latitude',
    'lon': 'longitude',
    'type': 'osm_type',
    'id': 'id'  # Keep OSM ID as 'id'
}).copy()

print("SPATIS fetched and loaded. Records:", len(spatis))
spatis.head(5)


SPATIS fetched and loaded. Records: 1607


,osm_type,id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,tags.amenity,tags.check_date:opening_hours,tags.compressed_air,tags.fuel:adblue,tags.fuel:biodiesel,tags.fuel:diesel,tags.fuel:e10,tags.fuel:octane_95,tags.fuel:octane_98,name,opening_hours,operator,tags.shop,tags.wheelchair,brand,tags.brand:wikidata,tags.brand:wikipedia,tags.fuel:GTL_diesel,tags.fuel:biogas,tags.fuel:cng,tags.fuel:lpg,tags.fuel:octane_102,tags.surveillance,website,tags.check_date,tags.dog,tags.email,tags.fax,phone,tags.start_date,tags.indoor_seating,tags.organic,tags.outdoor_seating,tags.smoking,tags.opening_hours:signed,tags.diet:halal,tags.level,tags.payment:credit_cards,tags.payment:debit_cards,tags.payment:apple_pay,tags.payment:cards,tags.payment:cash,tags.payment:google_pay,tags.payment:paypal,tags.drink:club-mate,tags.entrance,tags.FIXME,tags.created_by,tags.name:signed,tags.toilets:wheelchair,tags.wheelchair:description,tags.noname,tags.post_office,tags.post_office:service_provider,tags.lottery,tags.post_office:brand,tags.post_office:ref,tags.origin,tags.alt_name,tags.cuisine,tags.ref:vatin,tags.name:de,tags.name:en,tags.addr:floor,tags.air_conditioning,tags.contact:instagram,tags.contact:website,tags.internet_access,tags.stroller,tags.toilets,tags.access:covid19,tags.delivery:covid19,tags.post_office:type,tags.contact:fax,tags.contact:phone,tags.name:ru,tags.opening_hours:covid19,tags.takeaway:covid19,tags.internet_access:fee,tags.atm,tags.atm:operator,tags.note:de,tags.ref:Hermes,tags.description,tags.coffee,tags.tobacco,tags.post_office:brand:wikidata,tags.note,tags.operator:wikidata,tags.ref,tags.currency:EUR,source,tags.diet:gluten_free,tags.leisure,tags.cafe,tags.payment:coins,tags.vending:sweets,tags.vending:toys,tags.service:phone,tags.type,tags.diet:kosher,tags.diet:vegetarian,tags.post_box,tags.operator:wikipedia,tags.changing_table,tags.payment:mastercard,tags.payment:visa,tags.second_hand,tags.toilets:menstrual_products,tags.takeaway,tags.toilets:access,tags.parcel_pickup,tags.drink:coffee,tags.sells:tobacco,tags.name:zh,tags.addr:housename,tags.fixme,tags.contact:mobile,tags.dry_cleaning,tags.ice_cream,tags.drink:beer,tags.tickets:public_transport,tags.diet:vegan,tags.branch,tags.bulk_purchase,tags.diet:organic,tags.frozen_yogurt,tags.reusable_packaging:offer,tags.contact:email,tags.lgbtq,tags.post_office:letter,tags.post_office:parcel_to,tags.addr:place,tags.post_office:parcel_from,tags.post_office:parcel_pickup,tags.ref:deutsche_post,tags.tourism,tags.fair_trade,tags.payment:contactless,tags.payment:maestro,tags.payment:v_pay,tags.payment:app,tags.mapillary,tags.opening_date,tags.disused:name,tags.contact:facebook,tags.post_office:operator,tags.cash_in,tags.survey:date,tags.post_office:letter_from,tags.post_office:packaging,tags.post_office:stamps,tags.payment:girocard,tags.description:en,tags.old_name,tags.old_addr:housenumber,tags.old_addr:street,tags.reusable_packaging:accept,tags.wikidata,tags.wikipedia,tags.zero_waste,tags.not:brand:wikidata,tags.post_office:post_bank,tags.building,tags.roof:material,tags.roof:shape,tags.vending,tags.layer,tags.disused:shop,tags.service:copy,tags.service:fax,tags.service:scan,tags.self_checkout,tags.payment:american_express,tags.payment:notes,tags.payment:telephone_cards,tags.check_date:diet:gluten_free,tags.check_date:diet:halal,tags.payment:credit_cards:min_payment,tags.loc_name,tags.delivery,tags.bicycle,tags.craft,tags.indoor:level,tags.name:bg,tags.name:fa,tags.ramp:wheelchair,tags.toilets:charge,tags.smoking:outside,tags.description:name,tags.name:ar,tags.post_office:service_provider:wikidata,tags.wheelchair:description:de,tags.wheelchair:description:en,tags.payment:account_cards,tags.payment:alipay,tags.payment:bancomat,tags.payment:blik,tags.payment:cheque,tags.payment:clipper,tags.payment:cryptocurrencies,tags.payment:diners_club,tags.payment:discover_card,tags.payment:dkv,tags.payment:electronic_purse

In [351]:
print("Columns in spatis:", spatis.columns.tolist())

Columns in spatis: ['osm_type', 'id', 'latitude', 'longitude', 'tags.addr:city', 'tags.addr:country', 'tags.addr:housenumber', 'tags.addr:postcode', 'tags.addr:street', 'tags.addr:suburb', 'tags.amenity', 'tags.check_date:opening_hours', 'tags.compressed_air', 'tags.fuel:adblue', 'tags.fuel:biodiesel', 'tags.fuel:diesel', 'tags.fuel:e10', 'tags.fuel:octane_95', 'tags.fuel:octane_98', 'name', 'opening_hours', 'operator', 'tags.shop', 'tags.wheelchair', 'brand', 'tags.brand:wikidata', 'tags.brand:wikipedia', 'tags.fuel:GTL_diesel', 'tags.fuel:biogas', 'tags.fuel:cng', 'tags.fuel:lpg', 'tags.fuel:octane_102', 'tags.surveillance', 'website', 'tags.check_date', 'tags.dog', 'tags.email', 'tags.fax', 'phone', 'tags.start_date', 'tags.indoor_seating', 'tags.organic', 'tags.outdoor_seating', 'tags.smoking', 'tags.opening_hours:signed', 'tags.diet:halal', 'tags.level', 'tags.payment:credit_cards', 'tags.payment:debit_cards', 'tags.payment:apple_pay', 'tags.payment:cards', 'tags.payment:cash', 't

In [352]:
# Explore all columns and get summary statistics
spatis.describe(include="all")


,osm_type,id,latitude,longitude,tags.addr:city,tags.addr:country,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,tags.amenity,tags.check_date:opening_hours,tags.compressed_air,tags.fuel:adblue,tags.fuel:biodiesel,tags.fuel:diesel,tags.fuel:e10,tags.fuel:octane_95,tags.fuel:octane_98,name,opening_hours,operator,tags.shop,tags.wheelchair,brand,tags.brand:wikidata,tags.brand:wikipedia,tags.fuel:GTL_diesel,tags.fuel:biogas,tags.fuel:cng,tags.fuel:lpg,tags.fuel:octane_102,tags.surveillance,website,tags.check_date,tags.dog,tags.email,tags.fax,phone,tags.start_date,tags.indoor_seating,tags.organic,tags.outdoor_seating,tags.smoking,tags.opening_hours:signed,tags.diet:halal,tags.level,tags.payment:credit_cards,tags.payment:debit_cards,tags.payment:apple_pay,tags.payment:cards,tags.payment:cash,tags.payment:google_pay,tags.payment:paypal,tags.drink:club-mate,tags.entrance,tags.FIXME,tags.created_by,tags.name:signed,tags.toilets:wheelchair,tags.wheelchair:description,tags.noname,tags.post_office,tags.post_office:service_provider,tags.lottery,tags.post_office:brand,tags.post_office:ref,tags.origin,tags.alt_name,tags.cuisine,tags.ref:vatin,tags.name:de,tags.name:en,tags.addr:floor,tags.air_conditioning,tags.contact:instagram,tags.contact:website,tags.internet_access,tags.stroller,tags.toilets,tags.access:covid19,tags.delivery:covid19,tags.post_office:type,tags.contact:fax,tags.contact:phone,tags.name:ru,tags.opening_hours:covid19,tags.takeaway:covid19,tags.internet_access:fee,tags.atm,tags.atm:operator,tags.note:de,tags.ref:Hermes,tags.description,tags.coffee,tags.tobacco,tags.post_office:brand:wikidata,tags.note,tags.operator:wikidata,tags.ref,tags.currency:EUR,source,tags.diet:gluten_free,tags.leisure,tags.cafe,tags.payment:coins,tags.vending:sweets,tags.vending:toys,tags.service:phone,tags.type,tags.diet:kosher,tags.diet:vegetarian,tags.post_box,tags.operator:wikipedia,tags.changing_table,tags.payment:mastercard,tags.payment:visa,tags.second_hand,tags.toilets:menstrual_products,tags.takeaway,tags.toilets:access,tags.parcel_pickup,tags.drink:coffee,tags.sells:tobacco,tags.name:zh,tags.addr:housename,tags.fixme,tags.contact:mobile,tags.dry_cleaning,tags.ice_cream,tags.drink:beer,tags.tickets:public_transport,tags.diet:vegan,tags.branch,tags.bulk_purchase,tags.diet:organic,tags.frozen_yogurt,tags.reusable_packaging:offer,tags.contact:email,tags.lgbtq,tags.post_office:letter,tags.post_office:parcel_to,tags.addr:place,tags.post_office:parcel_from,tags.post_office:parcel_pickup,tags.ref:deutsche_post,tags.tourism,tags.fair_trade,tags.payment:contactless,tags.payment:maestro,tags.payment:v_pay,tags.payment:app,tags.mapillary,tags.opening_date,tags.disused:name,tags.contact:facebook,tags.post_office:operator,tags.cash_in,tags.survey:date,tags.post_office:letter_from,tags.post_office:packaging,tags.post_office:stamps,tags.payment:girocard,tags.description:en,tags.old_name,tags.old_addr:housenumber,tags.old_addr:street,tags.reusable_packaging:accept,tags.wikidata,tags.wikipedia,tags.zero_waste,tags.not:brand:wikidata,tags.post_office:post_bank,tags.building,tags.roof:material,tags.roof:shape,tags.vending,tags.layer,tags.disused:shop,tags.service:copy,tags.service:fax,tags.service:scan,tags.self_checkout,tags.payment:american_express,tags.payment:notes,tags.payment:telephone_cards,tags.check_date:diet:gluten_free,tags.check_date:diet:halal,tags.payment:credit_cards:min_payment,tags.loc_name,tags.delivery,tags.bicycle,tags.craft,tags.indoor:level,tags.name:bg,tags.name:fa,tags.ramp:wheelchair,tags.toilets:charge,tags.smoking:outside,tags.description:name,tags.name:ar,tags.post_office:service_provider:wikidata,tags.wheelchair:description:de,tags.wheelchair:description:en,tags.payment:account_cards,tags.payment:alipay,tags.payment:bancomat,tags.payment:blik,tags.payment:cheque,tags.payment:clipper,tags.payment:cryptocurrencies,tags.payment:diners_club,tags.payment:discover_card,tags.payment:dkv,tags.payment:electronic_purse

In [353]:
# Check missing values in each column
missing_count = spatis.isna().sum().sort_values(ascending=False)

# List columns where missing values are greater than 200
print(missing_count[missing_count > 200])


tags.fuel:HGV_diesel                     1606
tags.payment:credit_cards:min_payment    1606
tags.wikipedia                           1606
tags.post_office:post_bank               1606
tags.building                            1606
                                         ... 
tags.addr:postcode                        900
opening_hours                             889
tags.addr:housenumber                     812
tags.addr:street                          778
tags.wheelchair                           604
Length: 244, dtype: int64


In [354]:
# Check unique values in the 'brand' column
spatis['brand'].value_counts()



brand
REWE To Go                                  16
ServiceStore DB                             15
DHL                                          4
Yorma's                                      2
Total                                        2
Spar                                         2
DPD                                          1
JET                                          1
Shell Shop                                   1
Shell                                        1
Weltladen                                    1
Aral                                         1
Deutsche Post                                1
EDEKA                                        1
Elan                                         1
Lycamobile                                   1
Deutsche Post;DHL;Postbank;Western Union     1
Agip                                         1
Hermes                                       1
Edeka                                        1
TotalEnergies                                1
Name: c

In [355]:
# expand all columns to see more details
pd.set_option('display.max_columns', None)

print(spatis_gdf.head(3))


   type        id        lat        lon tags.addr:city tags.addr:country  \
0  node  26867411  52.501974  13.294496         Berlin                DE   
1  node  29997723  52.508370  13.280947         Berlin                DE   
2  node  63253672  52.499322  13.296118         Berlin                DE   

  tags.addr:housenumber tags.addr:postcode          tags.addr:street  \
0                    14              10711        Heilbronner Straße   
1                  8-10              14057                 Messedamm   
2                    39              10711  Joachim-Friedrich-Straße   

  tags.addr:suburb tags.amenity tags.check_date:opening_hours  \
0         Halensee         fuel                    2022-04-01   
1              NaN         fuel                           NaN   
2         Halensee          NaN                           NaN   

  tags.compressed_air tags.fuel:adblue tags.fuel:biodiesel tags.fuel:diesel  \
0                 yes              yes                 yes        

In [374]:
# Modelling & Planning
# Export SPATIS GeoDataFrame to the Downloads/Mapping folder

# Export as GeoJSON
spatis_gdf.to_file(
    "/Users/harrisongoodman/Downloads/spatis_raw.geojson",
    driver="GeoJSON"
)

# Export as CSV without geometry
spatis_gdf.drop(columns="geometry").to_csv(
    "/Users/harrisongoodman/Downloads/spatis_raw.csv",
    index=False
)

print("Raw SPATIS exported. Records:", len(spatis_gdf))


Raw SPATIS exported. Records: 1607


In [375]:
# Load districts
districts = gpd.read_file("/Users/harrisongoodman/Downloads/bezirksgrenzen.geojson")

# Rename correctly
districts = districts.rename(columns={
    "Gemeinde_schluessel": "district_id",
    "Gemeinde_name": "district_name"
})

# Keep only the necessary columns
districts = districts[["district_id", "district_name", "geometry"]].copy()

# Convert types
districts["district_id"] = districts["district_id"].astype(str)

print(districts.head())





  district_id               district_name  \
0         012               Reinickendorf   
1         004  Charlottenburg-Wilmersdorf   
2         009            Treptow-Köpenick   
3         003                      Pankow   
4         008                    Neukölln   

                                            geometry  
0  MULTIPOLYGON (((13.32074 52.6266, 13.32045 52....  
1  MULTIPOLYGON (((13.32111 52.52446, 13.32103 52...  
2  MULTIPOLYGON (((13.57925 52.39083, 13.57958 52...  
3  MULTIPOLYGON (((13.50481 52.6196, 13.50467 52....  
4  MULTIPOLYGON (((13.45832 52.48569, 13.45823 52...  


In [376]:
# Spatial join: assign each SPATIS point to a district
spatis = gpd.sjoin(
    spatis,
    districts[["district_id", "district_name", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")



In [377]:
# Load neighborhoods
neighborhoods = gpd.read_file("/Users/harrisongoodman/Downloads/lor_ortsteile.geojson")

# Correct: rename first
neighborhoods = neighborhoods.rename(columns={
    "spatial_name": "neighborhood_id",      # ID
    "OTEIL": "neighborhood_name"            # name
})

# Now select the correct columns
neighborhoods = neighborhoods[["neighborhood_id", "neighborhood_name", "geometry"]].copy()

# Convert ID to string
neighborhoods["neighborhood_id"] = neighborhoods["neighborhood_id"].astype(str)

# CRS fix to match SPATIS
neighborhoods = neighborhoods.to_crs(spatis.crs)

print(neighborhoods.head())

  neighborhood_id neighborhood_name  \
0            0101             Mitte   
1            0102            Moabit   
2            0103      Hansaviertel   
3            0104        Tiergarten   
4            0105           Wedding   

                                            geometry  
0  POLYGON ((13.41649 52.52696, 13.41635 52.52702...  
1  POLYGON ((13.33884 52.51974, 13.33884 52.51974...  
2  POLYGON ((13.34322 52.51557, 13.34323 52.51557...  
3  POLYGON ((13.36879 52.49878, 13.36891 52.49877...  
4  POLYGON ((13.34656 52.53879, 13.34664 52.53878...  


In [378]:

# Spatial join: neighborhoods

spatis = gpd.sjoin(
    spatis,
    neighborhoods[["neighborhood_id", "neighborhood_name", "geometry"]],
    how="left",
    predicate="within"
)

# Drop the index_right column added by sjoin
spatis = spatis.drop(columns=["index_right"], errors="ignore")

# Keep only the right-hand join columns and rename them
for col in ["neighborhood_id", "neighborhood_name"]:
    if f"{col}_right" in spatis.columns:
        spatis[col] = spatis[f"{col}_right"]

# Drop any leftover left/right duplicate columns
spatis = spatis.drop(columns=[
    "neighborhood_id_left", "neighborhood_id_right",
    "neighborhood_name_left", "neighborhood_name_right"
], errors="ignore")



In [379]:

if "district_id_right" in spatis.columns:
    spatis["district_id"] = spatis["district_id_right"]
    spatis["district_name"] = spatis["district_name_right"]
    spatis = spatis.drop(columns=[
        "district_id_left", "district_id_right",
        "district_name_left", "district_name_right"
    ], errors="ignore")



In [362]:

# --- District mapping (official Berlin codes as strings) ---
district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

# Apply mapping to create district_id
spatis['district_id'] = spatis['district_name'].map(district_mapping)

# (Optional) Check unmapped
unmapped = spatis[spatis['district_id'].isna()]['district_name'].unique()
if len(unmapped) > 0:
    print("⚠️ Unmapped districts:", unmapped)
else:
    print("✓ All districts mapped")




⚠️ Unmapped districts: [nan]


In [363]:
spatis = spatis.drop_duplicates(subset=["id"])

print("Duplicates removed. Records remaining:", len(spatis))


Duplicates removed. Records remaining: 1607


In [364]:

spatis = spatis.replace({None: np.nan})


In [365]:

print(spatis.columns.tolist())
print(spatis.head(3))


['osm_type', 'id', 'latitude', 'longitude', 'tags.addr:city', 'tags.addr:country', 'tags.addr:housenumber', 'tags.addr:postcode', 'tags.addr:street', 'tags.addr:suburb', 'tags.amenity', 'tags.check_date:opening_hours', 'tags.compressed_air', 'tags.fuel:adblue', 'tags.fuel:biodiesel', 'tags.fuel:diesel', 'tags.fuel:e10', 'tags.fuel:octane_95', 'tags.fuel:octane_98', 'name', 'opening_hours', 'operator', 'tags.shop', 'tags.wheelchair', 'brand', 'tags.brand:wikidata', 'tags.brand:wikipedia', 'tags.fuel:GTL_diesel', 'tags.fuel:biogas', 'tags.fuel:cng', 'tags.fuel:lpg', 'tags.fuel:octane_102', 'tags.surveillance', 'website', 'tags.check_date', 'tags.dog', 'tags.email', 'tags.fax', 'phone', 'tags.start_date', 'tags.indoor_seating', 'tags.organic', 'tags.outdoor_seating', 'tags.smoking', 'tags.opening_hours:signed', 'tags.diet:halal', 'tags.level', 'tags.payment:credit_cards', 'tags.payment:debit_cards', 'tags.payment:apple_pay', 'tags.payment:cards', 'tags.payment:cash', 'tags.payment:google_

In [380]:
selected_columns = [
    "id",
    "name",
    "brand",
    "operator",
    "latitude",
    "longitude",
    "district_id",
    "district_name",
    "neighborhood_id",
    "neighborhood_name",
    "openinghours",  
    "phone",
    "website",
    "geometry",
    "address"
]



In [367]:
final_cols = [
    "id", "name", "brand", "operator",
    "latitude", "longitude",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "openinghours",   # <-- correct name after renaming
    "phone", "website",
    "geometry",
    "address"
]





In [368]:
# Define the columns you want in the final output
final_cols = [
    "id", "name", "brand", "operator",
    "latitude", "longitude",
    "district_id", "district_name",
    "neighborhood_id", "neighborhood_name",
    "openinghours", "phone", "website",
    "geometry", "address"
]

# Keep only columns that exist in the joined spatis
final_cols = [c for c in final_cols if c in spatis.columns]

# Remove duplicate IDs
spatis = spatis.drop_duplicates(subset=["id"])

# Select the final columns
spatis = spatis[final_cols]

# Check result
spatis.head(5)








,id,name,brand,operator,latitude,longitude,district_id,district_name,neighborhood_id,neighborhood_name,phone,website,geometry
0,26867411,Bavaria petrol,NaN,Bavaria Petrol,52.501974,13.294496,11004004,Charlottenburg-Wilmersdorf,0407,Halensee,NaN,NaN,POINT (13.2945 52.50197)
1,29997723,Aral,Aral,Anne Notzke,52.508370,13.280947,11004004,Charlottenburg-Wilmersdorf,0405,Westend,NaN,https://tankstelle.aral.de/tankstelle/berlin/m...,POINT (13.28095 52.50837)
2,63253672,Späti Joe,NaN,NaN,52.499322,13.296118,11004004,Charlottenburg-Wilmersdorf,0407,Halensee,NaN,NaN,POINT (13.29612 52.49932)
3,253616592,Mein Markt Pham,NaN,NaN,52.511047,13.462698,11002002,Friedrichshain-Kreuzberg,0201,Friedrichshain,NaN,NaN,POINT (13.4627 52.51105)
4,266629404,...nah und gut,EDEKA,Heinz Vollack,52.509209,13.587500,11010010,Marzahn-Hellersdorf,1003,Kaulsdorf,+49 30 5677706,NaN,POINT (13.5875 52.50921)


In [346]:
# Count missing district_id and district_name
print("Missing district_id:", spatis['district_id'].isna().sum())
print("Missing district_name:", spatis['district_name'].isna().sum())

Missing district_id: 25
Missing district_name: 25


In [369]:
spatis = spatis.dropna(subset=['district_id', 'district_name'])
print("After dropping missing districts, rows left:", len(spatis))

After dropping missing districts, rows left: 1582


In [373]:

# Export final SPATIS dataset


# Export GeoJSON with geometry
spatis.to_file("spatis_with_admins.geojson", driver="GeoJSON")

# Export CSV without geometry
spatis.drop(columns="geometry").to_csv("spatis_with_admins.csv", index=False)

print("SPATIS export complete. Records:", len(spatis))


SPATIS export complete. Records: 1582


In [301]:
# Show all columns and their data types
spatis.dtypes.to_frame("dtype")


,dtype
id,int64
name,object
brand,object
operator,object
latitude,float64
longitude,float64
district_id,object
district_name,object
phone,object
website,object


In [ ]:
# Next Step: Step 2 — Fetch & Transform data.